# Phase 2b: fine-tune whisper-small with LoRA

Trains on the segmented pieces (`data/segments`, made by notebook 02) with `cyasr/train.py` and `configs/finetune_whisper-small.yaml`. Dev WER is checked every 100 steps; the best checkpoint is kept.

Run on a Colab T4 (Runtime > Change runtime type > T4 GPU). Before starting, run `bash scripts/make_colab_bundle.sh` locally and upload the new `cyasr_bundle.zip` to `MyDrive/cyasr/`, replacing the old one. The segments must already be in `MyDrive/cyasr/segments/`.

The real run writes straight to `MyDrive/cyasr/runs/` with `resume: true`: after a disconnect, reconnect and run the cells again (the smoke test can be skipped), and training continues from the last checkpoint. To start a run from scratch, delete or rename its folder on Drive first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
DRIVE_SEGMENTS = '/content/drive/MyDrive/cyasr/segments'
DRIVE_RUNS = '/content/drive/MyDrive/cyasr/runs'
CONFIG = 'configs/finetune_whisper-small.yaml'

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!ls cyasr/train.py {CONFIG}  # fails if the bundle is the old one
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch is already installed on Colab; this adds peft, the aligner and the other requirements
!pip install -q -r requirements.txt
# Colab ships an old torchao that peft rejects ("only versions above 0.16.0"); we don't use it
!pip uninstall -y -q torchao


In [ ]:
# Copy the segmented pieces from Drive to where train.py expects them
!mkdir -p data/segments && cp -r {DRIVE_SEGMENTS}/* data/segments/
import pandas as pd
seg = pd.read_csv('data/segments/segments.tsv', sep='\t')
print(seg.groupby('split').agg(pieces=('id', 'count'), hours=('duration_s', lambda s: round(s.sum() / 3600, 2))))
!ls data/segments/train | wc -l
!ls data/segments/dev | wc -l


In [ ]:
!mkdir -p data/segments
!rsync -r --info=progress2 {DRIVE_SEGMENTS}/ data/segments/
!ls data/segments/train | wc -l; ls data/segments/dev | wc -l

In [ ]:
!ls data/segments/train | wc -l; ls data/segments/dev | wc -l
!rsync -r --size-only --info=progress2 {DRIVE_SEGMENTS}/ data/segments/

In [ ]:
# Smoke test: same settings, but 4 steps and 8 dev pieces, saved locally (not on Drive).
# Checks that everything runs and how much GPU memory is used (peak_gpu_memory_gb in the timing line).
# The second run tests resume: it should print "resuming from .../checkpoint-4" and continue at step 5.
import yaml
!rm -rf runs/finetune_smoke
cfg = yaml.safe_load(open(CONFIG))
cfg.update(run_name='smoke', max_steps=4, eval_steps=2, warmup_steps=1, max_eval_samples=8)
yaml.safe_dump(cfg, open('/content/smoke.yaml', 'w'), allow_unicode=True)
!python -m cyasr.train --config /content/smoke.yaml
cfg.update(max_steps=6, resume=True)
yaml.safe_dump(cfg, open('/content/smoke.yaml', 'w'), allow_unicode=True)
!python -m cyasr.train --config /content/smoke.yaml


In [ ]:
# The real run, written straight to Drive with resume on. Roughly 40-60 min on an L4, up to ~2 h on a T4.
# After a disconnect: reconnect, run the cells above again (skip the smoke test), then this one.
import yaml
cfg = yaml.safe_load(open(CONFIG))
cfg.update(output_dir=DRIVE_RUNS, resume=True)
yaml.safe_dump(cfg, open('/content/run.yaml', 'w'), allow_unicode=True)
!mkdir -p {DRIVE_RUNS}
print('run folder:', f"{DRIVE_RUNS}/finetune_{cfg['run_name']}")
!python -m cyasr.train --config /content/run.yaml


In [ ]:
# Dev WER over training, and the time it took
import json, yaml
RUN = f"{DRIVE_RUNS}/finetune_" + yaml.safe_load(open(CONFIG))['run_name']
log = json.load(open(f'{RUN}/train_log.json'))
train_loss = {e['step']: e['loss'] for e in log if 'loss' in e}
print('step  train_loss  dev_loss  WER_basic  WER_nodiac  CER_nodiac')
for e in log:
    if 'eval_wer_basic' in e:
        tl = train_loss.get(e['step'], float('nan'))
        print(f"{e['step']:4}  {tl:10.3f}  {e['eval_loss']:8.3f}  {e['eval_wer_basic']:9.1%}  "
              f"{e['eval_wer_nodiac']:10.1%}  {e['eval_cer_nodiac']:10.1%}")
print(json.dumps(json.load(open(f'{RUN}/timing.json')), indent=2))


In [ ]:
# Everything is already on Drive. Check what is there and how big it is.
!ls {RUN}
!du -sh {RUN}/*
# Once best/ has been evaluated and you're happy with it, free the space taken by the checkpoints:
# !rm -rf {RUN}/checkpoints
